# 0. ETRI 위키백과 QA API 데이터 수집

본 노트북은 **ETRI 위키백과 QA API**를 사용하여 외부 QA 데이터셋을 수집하고 저장합니다.

**⚠️ 중요:**
- API 일일 호출 제한: 5,000건/일
- 수집된 데이터는 `data/etri_qa_dataset.json`에 저장됩니다
- 이 노트북은 **한 번만 실행**하면 됩니다
- 02, 03 노트북에서는 저장된 데이터를 불러와서 사용합니다


## 0.1. 환경 설정


In [ ]:
# 노트북 독립 실행을 위한 환경 설정
import sys
import os
from pathlib import Path

# 프로젝트 루트를 sys.path에 추가
project_root = Path().resolve().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 가상환경(.venv) 경로 추가
venv_path = project_root / ".venv"
if venv_path.exists():
    # Python 버전에 맞는 site-packages 경로 찾기
    python_version = f"{sys.version_info.major}.{sys.version_info.minor}"
    venv_site_packages = venv_path / "lib" / f"python{python_version}" / "site-packages"
    
    if venv_site_packages.exists():
        venv_path_str = str(venv_site_packages)
        if venv_path_str not in sys.path:
            sys.path.insert(0, venv_path_str)
        print(f"✅ 가상환경(.venv) 경로 추가됨: {venv_site_packages}")
    else:
        print(f"⚠️ 가상환경(.venv)이 존재하지만 site-packages를 찾을 수 없습니다")
else:
    print(f"ℹ️ 가상환경(.venv)이 없습니다. 시스템 Python을 사용합니다")

# 공통 유틸리티 import
try:
    from notebooks.utils import setup_notebook_environment, load_dataset_safely, load_json_safely
    
    # 환경 설정 (가상환경은 이미 위에서 설정했으므로 setup_venv=False)
    paths = setup_notebook_environment(setup_venv=False)
    print(f"✅ 프로젝트 루트: {paths['project_root']}")
    print(f"✅ 데이터 디렉토리: {paths['data_dir']}")
except ImportError as e:
    print(f"⚠️ 유틸리티 import 실패: {e}")
    paths = {
        'project_root': project_root,
        'data_dir': project_root / 'data',
        'notebook_dir': project_root / 'notebooks'
    }


import os
import sys
import json
import urllib3
import time
from pathlib import Path
from typing import Dict, List, Optional
from tqdm.auto import tqdm
from datasets import Dataset, DatasetDict, load_from_disk

# 프로젝트 루트 경로 설정
project_root = Path().resolve().parent.parent
sys.path.append(str(project_root))

# 데이터 저장 경로
data_dir = Path().resolve() / "data"
data_dir.mkdir(parents=True, exist_ok=True)

print(f"프로젝트 루트: {project_root}")
print(f"데이터 저장 경로: {data_dir}")


프로젝트 루트: /data/ephemeral/git/pro-nlp-mrc-nlp-01
데이터 저장 경로: /data/ephemeral/git/pro-nlp-mrc-nlp-01/notebooks/external_data_set/data


## 0.2. ETRI API 설정

⚠️ **API 키를 아래에 입력하세요!**


In [ ]:
# ETRI API 설정
ETRI_API_URL = "http://epretx.etri.re.kr:8000/api/WikiQA/"
# ⚠️ API 키는 환경 변수에서 가져오거나 직접 입력하세요
ETRI_ACCESS_KEY = os.getenv("ETRI_ACCESS_KEY", "")  # 환경 변수 또는 직접 입력

# 수집 설정
NUM_QUESTIONS = 2000  # 수집할 질문 수 (최대 5000/일)
API_DELAY = 0.2  # API 호출 간 딜레이 (초)

if not ETRI_ACCESS_KEY:
    print("⚠️ ETRI_ACCESS_KEY가 설정되지 않았습니다!")
    print("   환경 변수로 설정: export ETRI_ACCESS_KEY='your-key'")
    print("   또는 아래 셀에서 직접 입력하세요")

print(f"API URL: {ETRI_API_URL}")
print(f"수집 예정 질문 수: {NUM_QUESTIONS}")


API URL: http://epretx.etri.re.kr:8000/api/WikiQA/
수집 예정 질문 수: 2000
최소 신뢰도: 0.5


## 0.3. ETRI API 클라이언트


In [ ]:
class ETRIWikiQA:
    """ETRI 위키백과 QA API 클라이언트"""
    
    def __init__(self, access_key: str):
        self.api_url = ETRI_API_URL
        self.access_key = access_key
        self.http = urllib3.PoolManager(cert_reqs='CERT_NONE')
        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    
    def query(self, question: str, engine_type: str = "hybridqa") -> Dict:
        """위키백과 QA API 호출"""
        request_json = {
            "argument": {
                "question": question,
                "type": engine_type
            }
        }
        
        try:
            response = self.http.request(
                "POST",
                self.api_url,
                headers={
                    "Content-Type": "application/json; charset=UTF-8",
                    "Authorization": self.access_key
                },
                body=json.dumps(request_json)
            )
            
            if response.status == 200:
                return json.loads(response.data.decode('utf-8'))
            else:
                return None
                
        except Exception as e:
            return None
    
    def extract_qa_data(self, response: Dict) -> Optional[Dict]:
        """API 응답에서 QA 데이터 추출"""
        if not response or response.get('result') != 0:
            return None
        
        try:
            return_object = response.get('return_object', {})
            wiki_info = return_object.get('WiKiInfo', {})
            
            answer_info = wiki_info.get('AnswerInfo', [])
            if not answer_info:
                return None
            
            best_answer = answer_info[0]
            answer = best_answer.get('answer', '')
            confidence = best_answer.get('confidence', 0)
            
            ir_info = wiki_info.get('IRInfo', [])
            context = ""
            wiki_title = ""
            if ir_info:
                context = ir_info[0].get('sent', '')
                wiki_title = ir_info[0].get('wiki_title', '')
            
            return {
                'answer': answer,
                'confidence': confidence,
                'context': context,
                'wiki_title': wiki_title
            }
            
        except Exception as e:
            return None

# API 클라이언트 초기화
etri_qa = ETRIWikiQA(ETRI_ACCESS_KEY)
print("✅ ETRI API 클라이언트 초기화 완료")


✅ ETRI API 클라이언트 초기화 완료


## 0.4. API 연결 테스트


In [ ]:
# API 테스트
test_question = "대한민국의 수도는 어디인가요?"
print(f"테스트 질문: {test_question}")
print("-" * 50)

response = etri_qa.query(test_question)
if response:
    qa_data = etri_qa.extract_qa_data(response)
    if qa_data:
        print("✅ API 연결 성공!")
        print(f"정답: {qa_data['answer']}")
        print(f"신뢰도: {qa_data['confidence']:.4f}")
    else:
        print("❌ 데이터 추출 실패")
else:
    print("❌ API 연결 실패")


테스트 질문: 대한민국의 수도는 어디인가요?
--------------------------------------------------
✅ API 연결 성공!
정답: 서울특별시
신뢰도: 2.0427


## 0.5. 기존 데이터셋에서 질문 로드


In [ ]:
# 기존 데이터셋에서 질문 로드
original_data_path = project_root / "data" / "train_dataset"

if original_data_path.exists():
    original_datasets = load_from_disk(str(original_data_path))
    original_questions = original_datasets['train']['question']
    print(f"✅ 원본 데이터셋에서 {len(original_questions)}개의 질문 로드")
else:
    print("❌ 원본 데이터셋을 찾을 수 없습니다.")
    original_questions = []


✅ 원본 데이터셋에서 3952개의 질문 로드


## 0.6. ETRI API로 데이터 수집

⚠️ **이 셀은 시간이 오래 걸릴 수 있습니다 (약 3-5분)**


In [ ]:
def collect_etri_qa_data(
    questions: List[str],
    etri_client: ETRIWikiQA,
    num_questions: int = 1000,
    delay: float = 0.2
) -> List[Dict]:
    """
    ETRI API를 사용하여 QA 데이터 수집
    모든 질문-정답 페어를 저장합니다 (필터링 없음)
    """
    collected_data = []
    failed_count = 0
    
    sample_questions = questions[:num_questions]
    
    for idx, question in enumerate(tqdm(sample_questions, desc="데이터 수집")):
        try:
            response = etri_client.query(question, "hybridqa")
            
            if response:
                qa_data = etri_client.extract_qa_data(response)
                
                # 모든 질문-정답 페어를 저장 (필터링 없음)
                if qa_data and qa_data.get('answer'):
                    answer = qa_data['answer']
                    context = qa_data.get('context', '')
                    
                    # answer_start 찾기 (없으면 -1로 설정)
                    answer_start = context.find(answer) if context and answer else -1
                    
                    collected_data.append({
                        'id': f"etri-{idx:05d}",
                        'question': question,
                        'context': context,
                        'answers': {
                            'text': [answer] if answer else [],
                            'answer_start': [answer_start] if answer_start != -1 else []
                        },
                        'title': qa_data.get('wiki_title', ''),
                        'confidence': qa_data.get('confidence', 0.0)
                    })
                else:
                    failed_count += 1
            else:
                failed_count += 1
            
            time.sleep(delay)
            
        except Exception as e:
            failed_count += 1
            continue
    
    print(f"\n✅ 수집 완료!")
    print(f"   - 성공: {len(collected_data)}개")
    print(f"   - 실패/스킵: {failed_count}개")
    
    return collected_data

# 데이터 수집 실행
if original_questions and ETRI_ACCESS_KEY:
    print(f"📥 ETRI API로 {NUM_QUESTIONS}개의 질문에 대한 데이터 수집 시작...")
    print(f"   예상 소요 시간: 약 {NUM_QUESTIONS * API_DELAY / 60:.1f}분")
    print(f"   ⚠️ 모든 질문-정답 페어를 저장합니다 (필터링 없음)\n")
    
    etri_qa_data = collect_etri_qa_data(
        questions=original_questions,
        etri_client=etri_qa,
        num_questions=NUM_QUESTIONS,
        delay=API_DELAY
    )
elif not ETRI_ACCESS_KEY:
    print("❌ ETRI_ACCESS_KEY가 설정되지 않아 수집을 건너뜁니다.")
    etri_qa_data = []
else:
    print("❌ 질문 데이터가 없어 수집을 건너뜁니다.")
    etri_qa_data = []


📥 ETRI API로 2000개의 질문에 대한 데이터 수집 시작...
   예상 소요 시간: 약 6.7분



데이터 수집:   0%|          | 0/2000 [00:00<?, ?it/s]


✅ 수집 완료!
   - 성공: 408개
   - 실패/스킵: 1290개


## 0.7. 수집된 데이터 저장


In [ ]:
# 수집된 데이터 저장
etri_data_path = data_dir / "etri_qa_dataset.json"

if etri_qa_data:
    with open(etri_data_path, 'w', encoding='utf-8') as f:
        json.dump(etri_qa_data, f, ensure_ascii=False, indent=2)
    
    print(f"✅ 데이터 저장 완료!")
    print(f"   - 저장 경로: {etri_data_path}")
    print(f"   - 총 샘플 수: {len(etri_qa_data)}개")
else:
    print("❌ 저장할 데이터가 없습니다.")


## 0.8. 수집된 데이터 확인


In [ ]:
# 수집된 데이터 샘플 확인
if etri_qa_data:
    print("=== 수집된 데이터 샘플 ===\n")
    for i, sample in enumerate(etri_qa_data[:3]):
        print(f"[샘플 {i+1}]")
        print(f"  ID: {sample['id']}")
        print(f"  Question: {sample['question']}")
        print(f"  Answer: {sample['answers']['text'][0]}")
        print(f"  Context: {sample['context'][:100]}...")
        print(f"  Confidence: {sample['confidence']:.4f}")
        print()
    
    # 통계
    import numpy as np
    confidences = [d['confidence'] for d in etri_qa_data]
    print("=== 통계 ===")
    print(f"총 샘플 수: {len(etri_qa_data)}")
    print(f"평균 신뢰도: {np.mean(confidences):.4f}")
    print(f"최소 신뢰도: {np.min(confidences):.4f}")
    print(f"최대 신뢰도: {np.max(confidences):.4f}")
